# 自建GPT训练流程 - Part 5: 评估与对比

本notebook对整个训练流程中产生的三个模型进行系统评估和对比：

**设备建议：** CPU 可跑。

```
Base模型（预训练） → SFT模型（指令微调） → DPO模型（偏好对齐）
```

### 评估维度

| 维度 | 指标 | 说明 |
|------|------|------|
| **语言建模能力** | 困惑度 (Perplexity) | 越低越好 |
| **指令遵循能力** | 内容正确数 | 核对书况、处理动作和工作台 |
| **偏好一致性** | 内容及顺序均正确的记录数 | 在相同输入上检查动作是否在工作台之前 |

三个模型使用相同的提示格式和贪心解码。96 条测试记录来自 3 种留出的书况表述、8 个书名与 4 个来源的组合。

---

## 自建 GPT 训练流程

| 上一步 | 本步骤 | 完成！ |
|:---|:---|:---|
| ← [Part 4: DPO 对齐](./04_DPO_Training.ipynb) | **Part 5: 综合评估** | 训练流程完成 |

### 本步骤目标

1. 加载 Base / SFT / DPO 三个阶段的模型
2. 比较固定文本的困惑度、交接内容正确率和偏好顺序
3. 对照同一条书况记录上的三份模型回答
4. 理解全流程 Base → SFT → DPO 的能力演进

### 前置知识

- Part 1-4 的所有训练阶段
- Bonus B（评估方法论）


## 环境设置

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import os
import sys
import re
from collections import Counter

# 中文字体配置
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

# 兼容从项目根目录或本目录运行
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "Custom_GPT_Training" else os.getcwd()
CUSTOM_GPT_DIR = os.path.join(PROJECT_ROOT, "Custom_GPT_Training")
sys.path.insert(0, CUSTOM_GPT_DIR)

from custom_gpt import (
    CustomGPT, 
    GPTConfig, 
    SimpleTokenizer,
    count_parameters
)

# 设备选择
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

# 可复现性
torch.manual_seed(42)
np.random.seed(42)

使用设备: cuda


## Part 1: 加载所有模型

In [2]:
from handover import records, evaluate_replies
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "models", "library_handover")
tokenizer = SimpleTokenizer.load(os.path.join(CHECKPOINT_DIR, "tokenizer.pkl"))
models = {name: CustomGPT.from_pretrained(os.path.join(CHECKPOINT_DIR, path)).to(device)
          for name, path in [("预训练", "pretrained_model"), ("SFT", "sft_model"), ("DPO", "dpo_model")]}

Model loaded from .\models\library_handover\pretrained_model


Model loaded from .\models\library_handover\sft_model
Model loaded from .\models\library_handover\dpo_model


## Part 2: 困惑度评估 (Perplexity)

下面八句话用于比较三个阶段在同一段文本上的平均损失。预训练整份验证集的困惑度见 02；后面交接实验另用 96 条记录。

In [3]:
# 评估用测试文本
TEST_TEXTS = [
    "深度学习是机器学习的一个重要分支，使用多层神经网络来学习数据的表示。",
    "Transformer架构通过自注意力机制实现了并行计算，大大提高了训练效率。",
    "预训练语言模型首先在大规模语料上学习通用知识，然后针对具体任务进行微调。",
    "强化学习通过与环境的交互来学习最优策略，智能体通过奖励信号不断改进。",
    "注意力机制让模型能够动态关注输入的不同部分，是现代NLP的核心技术。",
    "梯度下降算法通过计算损失函数对参数的梯度来更新模型参数。",
    "过拟合是模型在训练数据上表现很好但在新数据上表现差的现象。",
    "正则化技术如Dropout和权重衰减可以有效防止模型过拟合。",
]

print(f"测试文本数量: {len(TEST_TEXTS)}")

测试文本数量: 8


In [4]:
@torch.no_grad()
def compute_perplexity(model, tokenizer, texts):
    """
    计算模型在文本集上的困惑度
    
    困惑度 = exp(平均交叉熵损失)
    """
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    for text in texts:
        # 编码
        ids = tokenizer.encode(text, add_bos=True, add_eos=True)
        if len(ids) < 2:
            continue
        
        # 截断到 max_seq_len 防止超出位置编码范围
        max_len = 128
        ids = ids[:max_len + 1]
        input_ids = torch.tensor([ids[:-1]]).to(device)
        labels = torch.tensor([ids[1:]]).to(device)
        
        # 前向传播
        outputs = model(input_ids)
        logits = outputs['logits']
        
        # 计算交叉熵
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            reduction='sum'
        )
        
        total_loss += loss.item()
        total_tokens += labels.numel()
    
    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    
    return perplexity, avg_loss

In [5]:
# 计算各模型的困惑度
print("模型困惑度对比:")
print("=" * 50)
print("提示: DPO优化人类偏好，困惑度可能上升，这里仍纳入对比。")

ppl_results = {}

for name, model in models.items():
    ppl, loss = compute_perplexity(model, tokenizer, TEST_TEXTS)
    ppl_results[name] = ppl
    print(f"{name}:")
    print(f"  困惑度: {ppl:.2f}")
    print(f"  平均Loss: {loss:.4f}")
    print()

模型困惑度对比:
提示: DPO优化人类偏好，困惑度可能上升，这里仍纳入对比。


预训练:
  困惑度: 316.09
  平均Loss: 5.7560

SFT:
  困惑度: 116317.68
  平均Loss: 11.6641



DPO:
  困惑度: 81936.65
  平均Loss: 11.3137



## Part 3: 指令遵循能力评估

In [6]:
cases = records(PROJECT_ROOT, "sft", "test")
predictions = {}
for name, candidate in models.items():
    correct, first, answers = evaluate_replies(candidate, tokenizer, cases)
    predictions[name] = answers
    print(f"{name}　内容正确 {correct}/{len(cases)}　其中动作在前 {first}/{len(cases)}")
pairs = records(PROJECT_ROOT, "dpo", "test")
gold = {row["prompt"]: row["chosen"] for row in pairs}
assert len(gold) == len(cases) and all(gold[row["instruction"]] == row["expected"] for row in cases)

预训练　内容正确 0/96　其中动作在前 0/96


SFT　内容正确 96/96　其中动作在前 0/96


DPO　内容正确 88/96　其中动作在前 88/96


In [7]:
for category in sorted({r["category"] for r in cases}):
    indices = [i for i, row in enumerate(cases) if row["category"] == category]
    # 若本类出现错例，优先展示，便于判断是内容错误还是顺序错误。
    i = next((j for j in indices if predictions["DPO"][j] != cases[j]["expected"]), indices[0])
    print("\n", cases[i]["instruction"], "\n参考交接条：", cases[i]["expected"])
    for name, answers in predictions.items():
        hits = sum(answers[j] == cases[j]["expected"] for j in indices)
        print(f"{name}　{answers[i]!r}　该类完全符合新交接习惯 {hits}/{len(indices)}")

# 保留同一本书、归还地点和书况，只替换明确的书况词。
from handover import reply
print("\nSFT 对同义表述的回答")
for status, paraphrase in [("缺页", "少了两页"), ("受潮", "湿了"), ("完好", "没有破损")]:
    i = next(i for i, row in enumerate(cases)
             if row["category"] == status and row["instruction"].startswith("西馆退回《旅行记》"))
    original = cases[i]["instruction"]
    revised = original.replace(status, paraphrase)
    print(original, "→", predictions["SFT"][i])
    print(revised, "→", reply(models["SFT"], tokenizer, revised))



 社区书屋退回《河流志》，归还前淋了雨，内页受潮。 
参考交接条： 受潮：风干，护理台
预训练　'                '　该类完全符合新交接习惯 0/32
SFT　'受潮：护理台，风干'　该类完全符合新交接习惯 0/32
DPO　'受潮：风干，护理台'　该类完全符合新交接习惯 32/32

 东馆退回《短篇集》，核过封面和内文，整本完好。 
参考交接条： 完好：上架，流通台
预训练　'                '　该类完全符合新交接习惯 0/32
SFT　'完好：流通台，上架'　该类完全符合新交接习惯 0/32
DPO　'完好：上架，流通台'　该类完全符合新交接习惯 32/32

 阅览室退回《旅行记》，前几章齐全，末章缺页。 
参考交接条： 缺页：补页，修书台
预训练　'                '　该类完全符合新交接习惯 0/32
SFT　'缺页：修书台，补页'　该类完全符合新交接习惯 0/32
DPO　'缺页，修书台'　该类完全符合新交接习惯 24/32

SFT 对同义表述的回答
西馆退回《旅行记》，前几章齐全，末章缺页。 → 缺页：修书台，补页
西馆退回《旅行记》，前几章齐全，末章少了两页。 → 缺页：修书台，补页
西馆退回《旅行记》，归还前淋了雨，内页受潮。 → 受潮：护理台，风干
西馆退回《旅行记》，归还前淋了雨，内页湿了。 → 缺页：修书台，补页
西馆退回《旅行记》，核过封面和内文，整本完好。 → 完好：流通台，上架
西馆退回《旅行记》，核过封面和内文，整本没有破损。 → 缺页：修书台，补页


## 结果分析

如果「动作在前」的次数增加了，「内容正确」的次数却减少了，回到三份回答里找：哪本书被送错了工作台，或是哪一步处理动作写错了？最后三组换说法的例子单独比较，没有计入这 96 条。

### 完整训练流程回顾

```
01_Model_Assembly.ipynb
├── 构建CustomGPT模型 (~2.73M参数)
├── RoPE位置编码 + RMSNorm + SwiGLU
└── 单句过拟合检查

02_Pretraining.ipynb
├── Next-token Prediction任务
├── 学习语言的统计规律
└── 输出: Base模型（会"说话"）

03_SFT_Training.ipynb
├── 按角色格式化的指令数据
├── Loss Masking（只对回答计算loss）
└── 输出: SFT模型（会"听话"）

04_DPO_Training.ipynb
├── (chosen, rejected)偏好对数据
├── 在冻结参考模型的基础上优化偏好对的对数概率比
└── 输出: DPO模型（会"选择"）

05_Evaluation.ipynb (本notebook)
├── 困惑度评估
├── 指令遵循能力
├── 同义表述对照
└── 偏好一致性
```
